## 模型評估與可解釋性 AI（SHAP 歸因與混淆矩陣分析）

- 目標：深入理解半導體極度不平衡數據下的模型評估策略，掌握為什麼在異常檢測中 召回率 (Recall) 遠比 準確率 (Accuracy) 重要。利用 Scikit-Learn 實作混淆矩陣與 Precision-Recall 曲線，並引入先進的可解釋性 AI 套件 SHAP，將黑盒子模型（如 XGBoost）的預測邏輯拆解為可視覺化的製程特徵歸因分析。


### 1. 異常檢測場景下的指標痛點：Recall 為什麼比 Accuracy 重要？

- 核心物理意義說明：在晶圓測試產線中，不良品（異常）通常僅佔整體的 1% ~ 2%。
    - 準確率 (Accuracy) 的陷阱：如果模型偷懶，把所有晶圓都猜「正常」，它的 Accuracy 依然高達 98% ~ 99%。但這樣模型根本沒有任何抓出瑕疵的能力。
    - 召回率 (Recall / 真正率)：代表「產線實際上的所有異常晶圓中，模型成功抓出了多少比例」。
    - 精準率 (Precision)：代表「模型報警的所有晶圓中，有多少比例是真正的異常」。
- 關鍵決策：在半導體產業中，放行一片漏電或有裂痕的潛在晶圓（未抓到異常，即偽陰性 False Negative），會導致客戶端退貨（RMA）並引發巨額賠償；相較之下，不小心把正常晶圓誤報為異常（偽陽性 False Positive），頂多由工程師手動複檢覆判。因此，寧可錯殺，不可放過，Recall 權重永遠高於 Accuracy。


### 2. 混淆矩陣與 Precision-Recall 曲線實作

- 實作：我們將模擬一個嚴重不平衡的晶圓缺陷分類結果，並繪製標準的評估指標。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    average_precision_score,
)

# 模擬產線不平衡資料：500 片正常 (0)，15 片異常 (1)
np.random.seed(42)
y_true = np.array([0] * 500 + [1] * 15)

# 模擬模型的預測機率（給予異常樣本較高的機率值，但也包含一些模糊地帶）
y_scores = np.concatenate(
    [
        np.random.uniform(0.0, 0.3, size=500),  # 正常樣本的預測機率
        np.random.uniform(0.4, 0.95, size=15),  # 異常樣本的預測機率
    ]
)

# 設定決策門檻值 (Threshold)，將機率轉為二元分類 (0 或 1)
# 為了追求高 Recall，我們將門檻值調低至 0.35（只要有 35% 機率是異常就報警）
threshold = 0.35
y_pred = (y_scores > threshold).astype(int)

# 計算並印出分類報告 (Classification Report)
print("=" * 60)
print("產線異常檢測模型評估報告：")
print(classification_report(y_true, y_pred, target_names=["Normal", "Anomaly"]))
print("=" * 60)

# 繪製混淆矩陣 (Confusion Matrix)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Reds",
    xticklabels=["猜正常 (0)", "猜異常 (1)"],
    yticklabels=["實際正常 (0)", "實際異常 (1)"],
)
plt.title("異常檢測：分類混淆矩陣 (Confusion Matrix)")
plt.ylabel("實際狀態")
plt.xlabel("模型預測")
plt.tight_layout()
plt.show()

### 3. 繪製 Precision-Recall 曲線 (PR Curve)

- 實作：對於極度不平衡的資料，ROC 曲線容易給出過於樂觀的假象，Precision-Recall 曲線 才是半導體異常檢測的標準規格。


In [ ]:
# 計算 PR 曲線的點位
precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)
avg_precision = average_precision_score(y_true, y_scores)

# 繪圖
plt.figure(figsize=(6, 4.5))
plt.plot(
    recalls,
    precisions,
    color="darkorange",
    lw=2,
    label=f"PR 曲線 (AP = {avg_precision:.3f})",
)
plt.xlabel("召回率 Recall (抓出異常的比例)")
plt.ylabel("精準率 Precision (預報警的準確度)")
plt.title("針對不平衡數據的 Precision-Recall 曲線")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

### 4. 可解釋性 AI：SHAP (SHapley Additive exPlanations) 歸因分析

- 實作：
    - 當 XGBoost 預測某片晶圓為異常時，製程工程師會追問原因。SHAP 利用賽局理論計算每個製程特徵對「預測值轉向異常」的邊際貢獻度（SHAP Value），讓 AI 具備物理說服力。
    - SHAP Summary Plot 中，橫軸代表對異常預測的貢獻大小，點的顏色代表特徵數值的高低。例如當 probe_temp_c 數值很高（紅色）且 SHAP 值為正時，直觀代表：高溫是引發模型報警的主要元凶。


In [ ]:
import xgboost as xgb
import shap

# 快速建立一個簡單的特徵矩陣 X 與模型進行演示
np.random.seed(42)
X_mock = pd.DataFrame(
    {
        "probe_temp_c": np.random.normal(45, 2, 515),
        "contact_force_g": np.random.normal(120, 10, 515),
        "bandwidth_ghz": np.random.normal(28, 1, 515),
    }
)

# 訓練一個臨時的 XGBoost 分類器
model_xgb = xgb.XGBClassifier(random_state=42)
model_xgb.fit(X_mock, y_true)

# 初始化 SHAP 解釋器 (TreeExplainer)
explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer(X_mock)

# 繪製特徵重要性蜂群圖 (Summary Plot)
print(">>> 正在生成 SHAP 歸因分析圖...")
plt.figure(figsize=(8, 4))
# SHAP 內部會自動繪圖，調整 matplotlib 視窗
shap.summary_plot(shap_values, X_mock, show=False)
plt.title("SHAP 可解釋性分析：特徵對晶圓異常預測的邊際貢獻度", fontsize=12)
plt.tight_layout()
plt.show()

- 總結：在半導體測試的實際場景中，資料分佈往往極度不平衡，可能一萬片晶圓中只有幾十片是真正的異常。因此，我在設計模型評估指標時，絕對不看 Accuracy，而是將核心聚焦在 Recall (召回率)。因為放行任何一片異常晶片所導致的 RMA（退貨）成本與商譽損失，遠大於誤報（False Alarm）的複檢成本。在專案中，我會透過 Precision-Recall 曲線 (PR Curve) 來動態調校最適合產線的決策門檻值。此外，為了解決機器學習黑盒子難以被製程工程師認可的痛點，我引入了 SHAP (可解釋性 AI) 機制。透過計算每個特徵的 Shapley 值，我可以向團隊精準指明：模型之所以判定這片晶圓異常，是因為探針床溫度 probe_temp_c 偏離了三個標準差。這種結合數據指標與物理可解釋性的能力，是讓機器學習能在測試產線真正落地的關鍵。
